# AI DevOps Reel Factory - Colab Runner

Use this notebook when the VPS does not have enough disk, RAM, or GPU support for XTTS, LivePortrait, Wav2Lip, and video rendering.

Before running: open `Runtime > Change runtime type`, select a T4 GPU, and connect the runtime. Colab is suitable for manual or semi-manual renders; it is not reliable as a permanent always-on scheduler.

## 1. Clone The Repo And Mount Drive

Google Drive is used for assets, model checkpoints, and outputs so the files survive Colab runtime resets.

Expected Drive layout:

```text
MyDrive/tutorlix_ai_devops_reel_factory/
  assets/
    face.png
    voice_sample.wav
    driving_videos/*.mp4
  LivePortrait/        # optional checkpoint/cache files copied into /content/LivePortrait
  Wav2Lip/             # optional wav2lip_gan.pth or checkpoints/wav2lip_gan.pth
  outputs/
```

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import json
import os
import shutil
import subprocess
from pathlib import Path

os.environ['MPLBACKEND'] = 'Agg'

REPO_URL = 'https://github.com/Xtute-Technologies/tutorlix_next.git'
BRANCH = 'main'
PROJECT_ROOT = Path('/content/tutorlix_next')
BOT_DIR = PROJECT_ROOT / 'bots' / 'ai_devops_reel_factory'
DRIVE_ROOT = Path('/content/drive/MyDrive/tutorlix_ai_devops_reel_factory')
VENV_DIR = Path('/content/ai-devops-py310')
LIVEPORTRAIT_VENV_DIR = Path('/content/liveportrait-py310')
PYTHON = VENV_DIR / 'bin' / 'python'
LIVEPORTRAIT_PYTHON = LIVEPORTRAIT_VENV_DIR / 'bin' / 'python'

for directory in [DRIVE_ROOT, DRIVE_ROOT / 'assets', DRIVE_ROOT / 'outputs', DRIVE_ROOT / 'LivePortrait', DRIVE_ROOT / 'Wav2Lip']:
    directory.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None):
    command = [str(part) for part in command]
    print('$', ' '.join(command))
    completed = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {' '.join(command)}")
    return completed

def ensure_pip(python_path):
    python_path = Path(python_path)
    probe = subprocess.run(
        [str(python_path), '-m', 'pip', '--version'],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )
    if probe.returncode == 0:
        print(probe.stdout.strip())
        return
    print(probe.stdout)
    print(f'pip is missing in {python_path}; bootstrapping with ensurepip')
    run([python_path, '-m', 'ensurepip', '--upgrade'])

def create_python310_venv(venv_dir, python_path):
    venv_dir = Path(venv_dir)
    python_path = Path(python_path)
    if not python_path.exists():
        run(['python', '-m', 'uv', 'python', 'install', '3.10'])
        run(['python', '-m', 'uv', 'venv', '--seed', '--python', '3.10', venv_dir])
    ensure_pip(python_path)

def colab_secret(name, default=''):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value if value is not None else default
    except Exception:
        return os.getenv(name, default)

def ensure_line(path, line):
    path = Path(path)
    text = path.read_text(encoding='utf-8') if path.exists() else ''
    lines = text.splitlines()
    if line not in lines:
        lines.insert(0, line)
        path.write_text('\n'.join(lines).rstrip() + '\n', encoding='utf-8')
        print(f'Patched {path}: added {line}')

def patch_script(path, replacements):
    path = Path(path)
    if not path.exists():
        return
    text = path.read_text(encoding='utf-8')
    original = text
    for old, new in replacements:
        if old in text and new not in text:
            text = text.replace(old, new, 1)
    if text != original:
        path.write_text(text, encoding='utf-8')
        print(f'Patched {path}')

def patch_wav2lip_repo():
    audio_py = Path('/content/Wav2Lip/audio.py')
    if audio_py.exists():
        text = audio_py.read_text(encoding='utf-8')
        original = text
        text = text.replace(
            'librosa.filters.mel(hp.sample_rate, hp.n_fft,',
            'librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft,',
        )
        if text != original:
            audio_py.write_text(text, encoding='utf-8')
            print('Patched Wav2Lip audio.py for modern librosa')

def liveportrait_required_weights(liveportrait_dir=Path('/content/LivePortrait')):
    root = Path(liveportrait_dir) / 'pretrained_weights'
    return [
        root / 'liveportrait' / 'base_models' / 'appearance_feature_extractor.pth',
        root / 'liveportrait' / 'base_models' / 'motion_extractor.pth',
        root / 'liveportrait' / 'base_models' / 'spade_generator.pth',
        root / 'liveportrait' / 'base_models' / 'warping_module.pth',
        root / 'liveportrait' / 'retargeting_models' / 'stitching_retargeting_module.pth',
        root / 'liveportrait' / 'landmark.onnx',
        root / 'insightface' / 'models' / 'buffalo_l' / '2d106det.onnx',
        root / 'insightface' / 'models' / 'buffalo_l' / 'det_10g.onnx',
    ]

def ensure_liveportrait_weights():
    missing = [path for path in liveportrait_required_weights() if not path.exists()]
    if not missing:
        print('LivePortrait pretrained weights found')
        return

    print('LivePortrait pretrained weights missing:')
    for path in missing:
        print(' -', path)

    # Official LivePortrait docs use:
    # huggingface-cli download KlingTeam/LivePortrait --local-dir pretrained_weights --exclude "*.git*" "README.md" "docs"
    # Use the Python API here because it gives clearer Colab errors and supports HF_TOKEN.
    run([LIVEPORTRAIT_PYTHON, '-m', 'pip', 'install', '--upgrade', 'huggingface_hub', 'hf_xet'])

    script = '''
from pathlib import Path
from huggingface_hub import snapshot_download
import os

local_dir = Path('/content/LivePortrait/pretrained_weights')
local_dir.mkdir(parents=True, exist_ok=True)
token = os.getenv('HF_TOKEN') or None
last_error = None

for repo_id, repo_type in [
    ('KlingTeam/LivePortrait', 'model'),
    ('KwaiVGI/LivePortrait', 'model'),
]:
    try:
        print(f'Downloading {repo_id} to {local_dir}')
        snapshot_download(
            repo_id=repo_id,
            repo_type=repo_type,
            local_dir=str(local_dir),
            local_dir_use_symlinks=False,
            token=token,
            ignore_patterns=['*.git*', 'README.md', 'docs/*'],
            resume_download=True,
        )
        print(f'Downloaded {repo_id}')
        break
    except Exception as exc:
        last_error = exc
        print(f'{repo_id} failed: {type(exc).__name__}: {exc}')
else:
    raise RuntimeError(f'Could not download LivePortrait weights from Hugging Face: {last_error}')
'''
    try:
        run([LIVEPORTRAIT_PYTHON, '-c', script])
    except RuntimeError as exc:
        raise RuntimeError(
            'LivePortrait weight download failed. If Hugging Face is rate-limited or blocked, add an HF_TOKEN Colab secret, '
            'or manually download KlingTeam/LivePortrait and copy liveportrait/ plus insightface/ into '
            '/content/drive/MyDrive/tutorlix_ai_devops_reel_factory/LivePortrait/pretrained_weights. '
            f'Original error: {exc}'
        )

    missing = [path for path in liveportrait_required_weights() if not path.exists()]
    if missing:
        raise FileNotFoundError(
            'LivePortrait weights are still missing after download. Missing: '
            + ', '.join(str(path) for path in missing)
        )
    print('LivePortrait pretrained weights ready')

def apply_colab_runtime_patches():
    requirements = BOT_DIR / 'requirements.txt'
    if requirements.exists():
        ensure_line(requirements, 'setuptools==80.9.0')

    voice_script = BOT_DIR / 'scripts' / 'generate_voice.py'
    if voice_script.exists():
        text = voice_script.read_text(encoding='utf-8')
        if 'os.environ["MPLBACKEND"] = "Agg"' not in text:
            marker = 'LOGGER = logging.getLogger("ai-devops-reel-factory.generate_voice")\n'
            patch = (
                'LOGGER = logging.getLogger("ai-devops-reel-factory.generate_voice")\n'
                '# Colab injects an inline matplotlib backend into the environment. Coqui TTS\n'
                '# imports matplotlib in a headless subprocess, so force a valid non-GUI backend.\n'
                'os.environ["MPLBACKEND"] = "Agg"\n'
            )
            if marker in text:
                text = text.replace(marker, patch, 1)
            else:
                text = 'import os\nos.environ["MPLBACKEND"] = "Agg"\n' + text
            voice_script.write_text(text, encoding='utf-8')
            print(f'Patched {voice_script}: forced MPLBACKEND=Agg')

    patch_script(
        BOT_DIR / 'scripts' / 'run_liveportrait.py',
        [
            (
                '    liveportrait_dir = config_path(config, "liveportrait_dir", must_exist=True)\n',
                '    liveportrait_dir = config_path(config, "liveportrait_dir", must_exist=True)\n    liveportrait_python = str(config.get("liveportrait_python") or python_executable())\n',
            ),
            (
                '        python_executable(),\n        "inference.py",',
                '        liveportrait_python,\n        "inference.py",',
            ),
        ],
    )
    patch_script(
        BOT_DIR / 'scripts' / 'run_wav2lip.py',
        [
            (
                '    wav2lip_dir = config_path(config, "wav2lip_dir", must_exist=True)\n',
                '    wav2lip_dir = config_path(config, "wav2lip_dir", must_exist=True)\n    wav2lip_python = str(config.get("wav2lip_python") or python_executable())\n',
            ),
            (
                '        python_executable(),\n        "inference.py",',
                '        wav2lip_python,\n        "inference.py",',
            ),
        ],
    )
    patch_wav2lip_repo()

github_token = colab_secret('GITHUB_TOKEN')
clone_url = REPO_URL
if github_token and REPO_URL.startswith('https://github.com/'):
    clone_url = REPO_URL.replace('https://', f'https://{github_token}@', 1)

if PROJECT_ROOT.exists():
    run(['git', 'fetch', 'origin'], cwd=PROJECT_ROOT)
    run(['git', 'checkout', BRANCH], cwd=PROJECT_ROOT)
    run(['git', 'pull', '--ff-only'], cwd=PROJECT_ROOT)
else:
    run(['git', 'clone', '--branch', BRANCH, clone_url, PROJECT_ROOT])
    if github_token:
        run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=PROJECT_ROOT)

apply_colab_runtime_patches()

print('Bot directory:', BOT_DIR)
print('Drive workspace:', DRIVE_ROOT)
print('Bot Python:', PYTHON)
print('LivePortrait Python:', LIVEPORTRAIT_PYTHON)


## 2. Install System And Python Dependencies

Colab runtimes may use Python 3.12, but Coqui `TTS==0.22.0`/XTTS is safest on Python 3.10. This cell creates a separate Python 3.10 environment with `uv` and installs the bot dependencies there.

This step can take several minutes and may download large GPU packages. If Colab asks to restart the runtime after installation, restart and rerun from step 1.


In [ ]:

run(['apt-get', 'update', '-qq'])
run([
    'apt-get', 'install', '-y', '-qq',
    'ffmpeg', 'git', 'fonts-dejavu-core', 'fonts-noto-core',
    'libgl1', 'libglib2.0-0', 'libsm6', 'libsndfile1', 'libxext6', 'libxrender1'
])

run(['python', '-m', 'pip', 'install', '--upgrade', 'pip', 'wheel', 'uv'])
create_python310_venv(VENV_DIR, PYTHON)

apply_colab_runtime_patches()
run([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip', 'wheel'])
run([PYTHON, '-m', 'pip', 'install', '-r', BOT_DIR / 'requirements.txt'])
run([PYTHON, '-m', 'pip', 'install', '--force-reinstall', 'setuptools==80.9.0'])
run([PYTHON, '-c', "import pkg_resources; print('pkg_resources ok')"])
run([PYTHON, '-V'])


## 3. Verify GPU

In [ ]:

run([
    PYTHON,
    '-c',
    "import torch; print('Torch:', torch.__version__); print('CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'); assert torch.cuda.is_available(), 'Colab is running without GPU. Change runtime type to T4 GPU and restart.'",
])


## 4. Sync Assets From Drive

Upload `face.png`, `voice_sample.wav`, and at least one driving video to the Drive `assets` folder before running this cell.

In [ ]:
ASSET_SOURCE = DRIVE_ROOT / 'assets'
ASSET_TARGET = BOT_DIR / 'assets'

required_files = [ASSET_SOURCE / 'face.png', ASSET_SOURCE / 'voice_sample.wav']
missing = [str(path) for path in required_files if not path.exists()]
driving_dir = ASSET_SOURCE / 'driving_videos'
driving_videos = [] if not driving_dir.exists() else [
    path for path in driving_dir.iterdir()
    if path.suffix.lower() in {'.mp4', '.mov', '.mkv', '.webm'}
]

if missing or not driving_videos:
    raise FileNotFoundError(
        'Missing Colab assets. Upload face.png, voice_sample.wav, and at least one video under '
        f'{ASSET_SOURCE}/driving_videos. Missing files: {missing}'
    )

shutil.copytree(ASSET_SOURCE, ASSET_TARGET, dirs_exist_ok=True)
print('Copied assets to', ASSET_TARGET)
print('Driving videos:', [path.name for path in driving_videos])

## 5. Install LivePortrait And Wav2Lip

The notebook clones the repos into `/content`. Put LivePortrait checkpoint files in the Drive `LivePortrait` folder and Wav2Lip checkpoint files in the Drive `Wav2Lip` folder; this cell merges them into the cloned repos.

In [ ]:

LIVEPORTRAIT_DIR = Path('/content/LivePortrait')
WAV2LIP_DIR = Path('/content/Wav2Lip')

if not (LIVEPORTRAIT_DIR / 'inference.py').exists():
    run(['git', 'clone', 'https://github.com/KwaiVGI/LivePortrait.git', LIVEPORTRAIT_DIR])

create_python310_venv(LIVEPORTRAIT_VENV_DIR, LIVEPORTRAIT_PYTHON)
run([LIVEPORTRAIT_PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip', 'wheel'])
run([LIVEPORTRAIT_PYTHON, '-m', 'pip', 'install', 'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1'])
run([LIVEPORTRAIT_PYTHON, '-m', 'pip', 'install', '-r', LIVEPORTRAIT_DIR / 'requirements.txt'], cwd=LIVEPORTRAIT_DIR)

liveportrait_drive = DRIVE_ROOT / 'LivePortrait'
if any(liveportrait_drive.iterdir()):
    shutil.copytree(liveportrait_drive, LIVEPORTRAIT_DIR, dirs_exist_ok=True)
    print('Merged LivePortrait files from Drive')
else:
    print('No Drive LivePortrait files found. Downloading official pretrained weights.')
ensure_liveportrait_weights()

if not (WAV2LIP_DIR / 'inference.py').exists():
    run(['git', 'clone', 'https://github.com/Rudrabha/Wav2Lip.git', WAV2LIP_DIR])

# Do not install Wav2Lip's requirements.txt. It pins numpy==1.17.1, which is
# incompatible with Python 3.10. The bot env already has compatible torch,
# opencv, scipy, librosa, numba, and tqdm. Patch the old librosa call instead.
patch_wav2lip_repo()
run([PYTHON, '-c', 'import torch, cv2, librosa, scipy, numba, tqdm; print("Wav2Lip compatible deps ok")'])
(WAV2LIP_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)

wav2lip_drive = DRIVE_ROOT / 'Wav2Lip'
if any(wav2lip_drive.iterdir()):
    shutil.copytree(wav2lip_drive, WAV2LIP_DIR, dirs_exist_ok=True)
    patch_wav2lip_repo()
    print('Merged Wav2Lip files from Drive')

checkpoint_candidates = [
    WAV2LIP_DIR / 'wav2lip_gan.pth',
    WAV2LIP_DIR / 'checkpoints' / 'wav2lip_gan.pth',
    WAV2LIP_DIR / 'checkpoints' / 'wav2lip.pth',
]
if not any(path.exists() for path in checkpoint_candidates):
    raise FileNotFoundError('Wav2Lip checkpoint missing. Put wav2lip_gan.pth in the Drive Wav2Lip folder.')


## 6. Configure Environment

Set `COQUI_TOS_AGREED=1` only after you have confirmed the Coqui XTTS license/TOS for your use case. For real Instagram publishing, also set the Instagram and public video URL secrets in Colab `Secrets` or assign them below. Google Drive share links are usually not suitable for `PUBLIC_VIDEO_BASE_URL`; Meta needs a direct public HTTPS MP4 URL.

In [ ]:
ENV_DEFAULTS = {
    'COQUI_TOS_AGREED': '0',
    'INSTAGRAM_DRY_RUN': 'true',
    'INSTAGRAM_USER_ID': '',
    'INSTAGRAM_ACCESS_TOKEN': '',
    'PUBLIC_VIDEO_BASE_URL': '',
    'MPLBACKEND': 'Agg',
    'HF_TOKEN': '',
}

for key, default in ENV_DEFAULTS.items():
    os.environ[key] = colab_secret(key, default)

if os.environ.get('COQUI_TOS_AGREED') != '1':
    raise RuntimeError('Set COQUI_TOS_AGREED=1 in Colab Secrets after confirming the Coqui XTTS license/TOS.')

config = json.loads((BOT_DIR / 'config.json').read_text())
config.update({
    'liveportrait_dir': str(LIVEPORTRAIT_DIR),
    'liveportrait_python': str(LIVEPORTRAIT_PYTHON),
    'wav2lip_dir': str(WAV2LIP_DIR),
    'wav2lip_python': str(PYTHON),
    'speaker_wav': str(ASSET_TARGET / 'voice_sample.wav'),
    'face_image': str(ASSET_TARGET / 'face.png'),
    'output_dir': str(DRIVE_ROOT / 'outputs'),
})

COLAB_CONFIG = BOT_DIR / 'config.colab.json'
COLAB_CONFIG.write_text(json.dumps(config, indent=2), encoding='utf-8')
print(COLAB_CONFIG.read_text())


## 7. Run The Pipeline

Use `render_only` first. Change to `dry_run_post` to exercise the Instagram posting code without calling Meta. Use `real_post` only after uploading `final_reel.mp4` to a public HTTPS host and setting `PUBLIC_VIDEO_BASE_URL`.

In [ ]:

RUN_MODE = 'render_only'  # render_only, dry_run_post, real_post

command = [PYTHON, 'scripts/main.py', '--config', COLAB_CONFIG]
if RUN_MODE == 'render_only':
    command.append('--skip-post')
elif RUN_MODE == 'dry_run_post':
    command.append('--dry-run-post')
elif RUN_MODE == 'real_post':
    os.environ['INSTAGRAM_DRY_RUN'] = 'false'
else:
    raise ValueError(f'Unknown RUN_MODE: {RUN_MODE}')

run(command, cwd=BOT_DIR)
print('Final reel:', DRIVE_ROOT / 'outputs' / 'final_reel.mp4')


## Repair Current Runtime

Run this cell if you already installed dependencies and hit `pkg_resources` or matplotlib backend errors. It patches the cloned repo and the Python 3.10 environment without restarting Colab.


In [ ]:

os.environ['MPLBACKEND'] = 'Agg'
apply_colab_runtime_patches()
ensure_pip(PYTHON)
run([PYTHON, '-m', 'pip', 'install', '--force-reinstall', 'setuptools==80.9.0'])
run([PYTHON, '-c', "import pkg_resources; from TTS.api import TTS; print('TTS import ok')"])
patch_wav2lip_repo()
if LIVEPORTRAIT_PYTHON.exists() and Path('/content/LivePortrait').exists():
    ensure_liveportrait_weights()


## Troubleshoot Missing `final_reel.mp4`

If the pipeline stops after creating only `script.txt` and `script_metadata.json`, run this cell. It prints the output checklist and reruns the first missing stage with full logs.


In [ ]:

EXPECTED_OUTPUTS = [
    'script.txt',
    'script_metadata.json',
    'voice.wav',
    'liveportrait.mp4',
    'synced_face.mp4',
    'final_reel.mp4',
]

STAGE_PLAN = [
    ('voice.wav', ['scripts/generate_voice.py', '--config', COLAB_CONFIG]),
    ('liveportrait.mp4', ['scripts/run_liveportrait.py', '--config', COLAB_CONFIG]),
    ('synced_face.mp4', ['scripts/run_wav2lip.py', '--config', COLAB_CONFIG]),
    ('final_reel.mp4', ['scripts/render_reel.py', '--config', COLAB_CONFIG]),
]

def output_checklist():
    outputs = DRIVE_ROOT / 'outputs'
    print('Outputs directory:', outputs)
    outputs.mkdir(parents=True, exist_ok=True)
    for name in EXPECTED_OUTPUTS:
        path = outputs / name
        status = 'OK' if path.exists() and path.stat().st_size > 0 else 'MISSING'
        size = path.stat().st_size if path.exists() else 0
        print(f'{status:8} {name:22} {size} bytes')


def run_stage_with_logs(label, args):
    command = [PYTHON, *args]
    print('\nRunning missing stage for', label)
    print('$', ' '.join(str(part) for part in command))
    completed = subprocess.run(
        [str(part) for part in command],
        cwd=str(BOT_DIR),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )
    print(completed.stdout)
    output_checklist()
    if completed.returncode != 0:
        raise RuntimeError(f'{label} stage failed with exit code {completed.returncode}. Read the log above.')

output_checklist()
for output_name, args in STAGE_PLAN:
    output_path = DRIVE_ROOT / 'outputs' / output_name
    if not output_path.exists() or output_path.stat().st_size == 0:
        run_stage_with_logs(output_name, args)
        break
else:
    print('All expected outputs exist. Preview final_reel.mp4 in the next cell.')


## 8. Preview The Output

In [ ]:
from IPython.display import Video, display

final_video = DRIVE_ROOT / 'outputs' / 'final_reel.mp4'
if not final_video.exists():
    raise FileNotFoundError(final_video)

print(final_video)
display(Video(str(final_video), width=360, embed=False))

## Optional: Run The Scheduler While Colab Is Open

Colab can disconnect at any time. Use this only for an active session, not as production cron.

In [ ]:

# run([PYTHON, 'scripts/scheduler.py', '--config', COLAB_CONFIG, '--run-now', '--dry-run-post'], cwd=BOT_DIR)
